# 反爬：登入授權模擬

* 了解「登入權限機制」的反爬蟲機制
* 「登入權限機制」反爬蟲的因應策略

## 作業目標

* 找一個需要登入的網站試試看，並說明思考流程
（如果不知道要用哪個網站的話，可以試試看 https://github.com/new 網址，未登入時會被導向登入頁）



In [ ]:
!pip install -U selenium
!pip install webdriver_manager
!pip install fake-useragent
!pip install undetected-chromedriver


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import time

# 建立 Chrome 瀏覽器物件
driver = webdriver.Chrome()
driver.maximize_window()


In [ ]:
import time
import pickle
import undetected_chromedriver as uc

# 建立瀏覽器，開啟 Shopee 首頁
driver = uc.Chrome()
driver.get("https://shopee.tw/")

print("請在此瀏覽器視窗中手動登入 Shopee（例如掃碼登入）。")
input("登入完成後請按 Enter...")

# 取得登入後的 Cookies
cookies = driver.get_cookies()
with open("shopee_cookies.pkl", "wb") as f:
    pickle.dump(cookies, f)
print("✅ Cookies 已成功儲存到 shopee_cookies.pkl")

driver.quit()

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pickle
import time
import pandas as pd

# === 1. 設定瀏覽器 ===
options = Options()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(options=options)

# === 2. 先進入首頁，載入 cookie ===
driver.get("https://shopee.tw/")
time.sleep(2)

with open("shopee_cookies.pkl", "rb") as f:
    cookies = pickle.load(f)
    for cookie in cookies:
        if "expiry" in cookie:
            del cookie["expiry"]
        driver.add_cookie(cookie)

# === 3. 重新導向搜尋頁面 ===
driver.get("https://shopee.tw/search?keyword=滑鼠")
time.sleep(5)

# === 4. 緩慢下滑，讓商品載入 ===
last_height = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollBy(0, 1000);")
    time.sleep(1.5)
    new_height = driver.execute_script("return document.body.scrollHeight")
    if new_height == last_height:
        break
    last_height = new_height

# === 5. 等商品名稱區塊出現 ===
WebDriverWait(driver, 15).until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.line-clamp-2.text-sm"))
)

# === 6. 抓取商品名稱（略過圖片）===
elements = driver.find_elements(By.CSS_SELECTOR, "div.line-clamp-2.text-sm")

results = []
for el in elements:
    # 只取純文字部分，不包括圖片
    text_nodes = driver.execute_script("""
        let el = arguments[0];
        let texts = [];
        for (let node of el.childNodes) {
            if (node.nodeType === Node.TEXT_NODE) {
                texts.push(node.textContent.trim());
            }
        }
        return texts.join(' ');
    """, el)

    if text_nodes:
        results.append({"商品名稱": text_nodes})

# === 7. 存成 CSV ===
df = pd.DataFrame(results)
df.to_csv("shopee_滑鼠商品名稱.csv", index=False, encoding="utf-8-sig")
print("✅ 共儲存", len(df), "筆商品名稱")
driver.quit()